# 📗 그래프에 문서 얹기: 임베딩 적재와 벡터·전문 인덱스

지금까지 그래프에서 원하는 것을 찾을 때는 **정확한 조건**(레이블·속성·패턴)으로 조회했습니다. 그런데 사람의 질문은 대개 **표현이 다릅니다**. "어떤 유전자가 약을 분해하나"라고 물어도 논문 제목은 "Pharmacogenomic landscape"일 수 있죠. 글자가 겹치지 않으면 조건 검색은 놓칩니다.

이번 단원에서는 **의학 논문을 지식그래프 안에 넣습니다.** 논문을 `:Document` 노드로 올리고, 임베딩을 붙여 **뜻으로 찾고**, 그 논문이 언급한 약물·유전자와 이어 **그래프로 넘어갑니다.** 이것이 이 교과목이 목표로 하는 GraphRAG 의 바탕입니다.

## ⏪ 복습: 지난 시간까지

- **임베딩**: 문장을 **고정 길이 숫자 벡터**로 바꾸면, 뜻이 비슷한 문장은 벡터도 가까워집니다. 가까운 정도는 **코사인 유사도**(1에 가까울수록 비슷)로 잽니다.
- **지식그래프**: Hetionet 의료 그래프를 적재하고(노드 5종·관계 12종), Cypher 로 조회했습니다. 오늘 **1-1** 에서 그 파일을 그대로 다시 올립니다.
- **GDS**: **PageRank**(중요한 노드)와 **커뮤니티**(촘촘히 이어진 묶음)를 구하는 법을 배웠습니다. 다음 노트북에서 **같은 방법으로 다시 계산해** 검색 순위에 씁니다(33·34일차의 결과 파일을 옮겨 오는 것이 아닙니다).
- 오늘은 이 셋을 잇습니다. **논문을 그래프에 얹고**(2절), **벡터·전문 인덱스로 찾고**(3·4절), **개체와 이어 둡니다**(5절).

**오늘의 목표**

**1. 그래프와 논문을 한자리에**
- [ ] (1-1) **32일차에 만든 지식그래프**를 다시 올리고 무엇이 들어 있는지 훑는다.
- [ ] (1-2) 오늘 얹을 **논문 69편**이 어떤 글인지 열어 보고, 오늘 만들 모양을 그림으로 잡는다.
- [ ] (1-3) **임베딩**이 768차원·길이 1 로 나오는지 확인하고, 오늘 새로운 것 하나를 가른다.

**2. 임베딩 적재**
- [ ] (2-1) 문서 임베딩을 **노드 속성으로 적재**한다(`db.create.setNodeVectorProperty`).

**3. 벡터 인덱스와 의미 검색**
- [ ] (3-1) **벡터 인덱스**를 만들고(차원 768·유사도 `cosine` 을 적어 주는 것이 핵심) 그 값을 `SHOW VECTOR INDEXES` 로 확인한다.
- [ ] (3-2) **`SEARCH` 절**로 의미 검색하고, `LIMIT` 이 거르기 전에 자른다는 것과 그 대처 둘을 눈에 담는다.

**4. 전문 인덱스: 키워드로 찾기**
- [ ] (4-1) **전문(full-text) 인덱스**를 만들어 키워드로 찾고, 검색어 문법(`AND`·`OR`·`title:`)을 써 본다.
- [ ] (4-2) **의미 검색과 글자 검색이 갈리는 자리**를 설명한다.
- [ ] (4-3) 하이픈이 든 기호는 **따옴표로 묶어야** 하는 까닭을 안다.

**5. 문서를 개체에 잇기**
- [ ] (5-1) 이름을 **`id` 로 바꾸는 사전**으로 논문에서 개체를 찾는다.
- [ ] (5-2) **`MENTIONS`** 로 잇고, 다리가 정말 놓였는지 검산한다.
- [ ] (5-3) 문서에서 **그래프로 한 걸음** 건너가고, 이 잇기가 무엇을 놓치는지 말한다.

> **데이터 출처**: 이 단원의 데이터는 **공개된 원본을 값 그대로** 쓴 것입니다.
>
> | 데이터 | 원본 | 이용 조건 |
> |---|---|---|
> | 논문 69편 (`pmc_docs.jsonl`) | PubMed Central Open Access Subset. 각 행의 `pmcid` 가 원문 주소다 | **CC BY** |
> | 의료 지식그래프 (`hetionet_*.csv`) | Hetionet v1.0 (https://het.io) 에서 CC0 출처만 골라낸 부분 | **CC0** |
> | 이름 사전 (`name2id.json.gz`) | 위 Hetionet 이름 + RxNav(미국 국립의학도서관) 약물 동의어 | CC0 · NLM |
>
> 지식그래프는 **2016년에 정리된 자료**이고, 논문은 최근 것입니다. 그래서 이 둘을 이어 붙이면 그래프가 모르는 사실이 논문 쪽에 있습니다. 이 단원은 그 상태 그대로 검색합니다.
>
> 그리고 **논문이 보고했다**와 **효능이 입증됐다**는 다릅니다. 검색으로 찾은 문장을 답으로 옮길 때 이 구분을 놓치면, 근거가 있는 것처럼 보이는 틀린 답이 나옵니다.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 임베딩 준비: embed_texts(문장 리스트) 가 768차원 OpenAI 임베딩을 돌려줍니다(실행만 하세요).
# embed_texts(문서 리스트) 는 계산한 벡터를 data/emb_cache.pkl 에 저장해 두고 다시 씁니다(69편을 매번 다시 부르지 않으려고요).
# embed_query(질문 한 문장) 는 저장하지 않습니다. 질문은 매번 새로 만드는 것이 표준입니다.
import pickle
from pathlib import Path

from langchain_openai import OpenAIEmbeddings

EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM = 768                    # 3072차원으로 나오는 모델을 768차원으로 잘라 받는다(아래 dimensions)
_candidates = [Path("data/emb_cache.pkl"), Path("내작업폴더/day35_벡터검색_GraphRAG/data/emb_cache.pkl"), Path("day35_벡터검색_GraphRAG/data/emb_cache.pkl"), Path("../data/emb_cache.pkl")]
_EMB_FILE = next((p for p in _candidates if p.exists()), Path("data/emb_cache.pkl"))
_EMB_CACHE = pickle.loads(_EMB_FILE.read_bytes()) if _EMB_FILE.exists() else {}   # {문서: 벡터}
embedder = OpenAIEmbeddings(model=EMBED_MODEL, dimensions=EMBED_DIM)


def embed_texts(texts):
    """문서 리스트 -> 768차원 임베딩 리스트. 저장된 것은 그대로 쓰고, 없는 것만 임베딩해 저장한다."""
    new = [t for t in texts if t not in _EMB_CACHE]                  # 저장돼 있지 않은 문서만 고른다
    if new:
        _EMB_CACHE.update(zip(new, embedder.embed_documents(new)))   # 실제 호출은 이 줄뿐
        _EMB_FILE.write_bytes(pickle.dumps(_EMB_CACHE))              # 통째로 다시 저장
    return [_EMB_CACHE[t] for t in texts]


def embed_query(text):
    """질문 한 문장 -> 768차원 임베딩. 질문은 저장하지 않는다(매번 새로 만든다)."""
    return embedder.embed_query(text)


print("임베딩 모델:", EMBED_MODEL, f"({EMBED_DIM}차원) / 저장된 문서:", len(_EMB_CACHE), "건")

In [ ]:
# [제공 코드] Neo4j 연결: 변경 없이 그대로 실행하세요.
# - 로컬 실습 전용(포트 7689)으로 안전하게 연결됩니다.
# - 클라우드 Aura DB 접속은 가드에 의해 100% 원천 차단됩니다.
import os
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 현재 폴더의 .env를 override=True로 강제 로드
env_candidates = [
    Path('.env'),
    Path('day35_벡터검색_GraphRAG/.env'),
    Path('내작업폴더/day35_벡터검색_GraphRAG/.env'),
    Path('../.env')
]
for p in env_candidates:
    if p.exists():
        load_dotenv(p, override=True)

# 로컬 실습 기본값 강제 (포트 7689)
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7689')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'test0011')

# 🚨 [안전 가드] 클라우드 DB 접속 원천 차단
if 'databases.neo4j.io' in str(NEO4J_URI):
    # 만약 환경변수에 클라우드가 남아있더라도 강제로 로컬 7689로 전환
    print('⚠️ 클라우드 Aura 주소가 감지되어 로컬 bolt://localhost:7689 로 자동 강제 전환합니다.')
    NEO4J_URI = 'bolt://localhost:7689'
    NEO4J_USER = 'neo4j'
    NEO4J_PASSWORD = 'test0011'

# 2) 드라이버 연결
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

# 3) Cypher 실행 헬퍼
def run_cypher(query, **params):
    with driver.session() as session:
        return [record.data() for record in session.run(query, **params)]

print('Neo4j 연결:', NEO4J_URI)


In [ ]:
# [제공 코드] 실습 전용 DB 초기화: 이 셀은 실행만 하세요.
# 몇 번이든 처음부터 다시 돌릴 수 있게 전부 내립니다. 반드시 "실습 전용" DB 여야 합니다.

# 1) GDS 투영: 노드를 지우기 전에 먼저 내린다. 투영은 원본 노드 id 를 기억하고 있어, 원본을 먼저 지우면 갈 곳을 잃는다
for _g in run_cypher("CALL gds.graph.list() YIELD graphName RETURN graphName"):
    run_cypher("CALL gds.graph.drop($name) YIELD graphName RETURN graphName", name=_g["graphName"])

# 2) 노드와 관계
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH: 노드에 붙은 관계까지 함께 지운다

# 3) 벡터·전문 인덱스: 노드를 지워도 인덱스는 남는다. 차원이 다른 옛 인덱스가 남아 있으면 뒤에서 걸린다
for _ix in run_cypher("SHOW INDEXES YIELD name, type WHERE type IN ['VECTOR','FULLTEXT'] RETURN name"):
    run_cypher(f"DROP INDEX {_ix['name']} IF EXISTS")   # IF EXISTS: 이미 없어도 에러 없이 넘어간다

print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

---
# 1. 그래프와 논문을 한자리에

오늘 다룰 재료 둘을 한자리에 놓고 시작합니다. 32일차에 만든 **지식그래프**와, 오늘 새로 얹을 **논문 69편**입니다. 코드를 쓰기 전에 둘이 각각 어떤 물건인지부터 봅니다.

- **1-1** 32일차의 지식그래프를 다시 올리고 무엇이 들어 있는지 훑습니다.
- **1-2** 오늘 얹을 논문 69편을 열어 보고, 오늘 만들 모양을 그림으로 잡습니다.
- **1-3** 임베딩이 768차원·길이 1 로 나오는지 확인합니다.

## 1-1. 32일차의 지식그래프를 다시 올리기

### 왜 필요할까요?
- **지식그래프**: 약물·질병·유전자·증상·약효분류가 관계로 이어져 있습니다. 사실이 **정리된** 형태입니다.
- **논문**: 사람이 쓴 문장 덩어리입니다. 사실이 **문장 속에** 있습니다.

그래프는 조회가 정확하지만 **2016년에 정리된 것만** 압니다. 논문은 최신이지만 **찾기가 어렵습니다.** 오늘 하는 일은 논문을 그래프 안에 넣어 두 성질을 함께 쓰는 것입니다.

먼저 그래프를 올립니다(32일차에서 쓰던 그 파일입니다).

In [ ]:
# [제공 코드] 의료 지식 그래프 적재: 이 셀은 실행만 하세요(2초쯤 걸립니다).
# 32일차에서 적재한 그 그래프입니다(Hetionet v1.0 중 재배포 가능한 CC0 부분, 2016년 자료).
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")   # 정답 폴더에서도 돌게
# 이 그래프의 노드 레이블 5종: 약물·질병·유전자·증상·약효분류
NODE_LABELS = ['Compound', 'Disease', 'Gene', 'Symptom', 'PharmacologicClass']
# 관계 이름 -> (출발 레이블, 도착 레이블). 아래 MATCH 에 레이블을 찍어 인덱스를 타게 하려고 미리 적어 둔다
REL_ENDS = {
    "TREATS": ("Compound", "Disease"),
    "PALLIATES": ("Compound", "Disease"),
    "BINDS": ("Compound", "Gene"),
    "UPREGULATES_CG": ("Compound", "Gene"),
    "DOWNREGULATES_CG": ("Compound", "Gene"),
    "ASSOCIATES": ("Disease", "Gene"),
    "UPREGULATES_DG": ("Disease", "Gene"),
    "DOWNREGULATES_DG": ("Disease", "Gene"),
    "RESEMBLES_CC": ("Compound", "Compound"),
    "RESEMBLES_DD": ("Disease", "Disease"),
    "PRESENTS": ("Disease", "Symptom"),
    "INCLUDES": ("PharmacologicClass", "Compound"),
}

# 1) id 인덱스부터: 관계를 만들 때 노드를 id 로 찾으므로, 없으면 매번 전수 스캔이 된다
for _label in NODE_LABELS:
    run_cypher(f"CREATE INDEX {_label.lower()}_id IF NOT EXISTS FOR (n:{_label}) ON (n.id)")

# 2) 노드 적재: csv 를 레이블별로 나눠 담고 레이블마다 한 번에 보낸다
_nodes = {_label: [] for _label in NODE_LABELS}
for _row in pd.read_csv(DATA_DIR / "hetionet_nodes.csv").to_dict("records"):
    _nodes[_row["label"]].append({"id": _row["id"], "name": _row["name"]})
for _label, _rows in _nodes.items():
    run_cypher(f"UNWIND $rows AS row CREATE (n:{_label}) SET n.id = row.id, n.name = row.name",
               rows=_rows)

# 3) 관계 적재: 2만 건씩 끊어 보낸다(한 번에 다 보내면 메모리를 많이 쓴다)
_edges = {_rel: [] for _rel in REL_ENDS}
for _row in pd.read_csv(DATA_DIR / "hetionet_edges.csv").to_dict("records"):
    _edges[_row["rel"]].append({"s": _row["source"], "t": _row["target"]})
for _rel, _rows in _edges.items():
    _src, _dst = REL_ENDS[_rel]
    for _start in range(0, len(_rows), 20000):
        run_cypher(f"UNWIND $rows AS row "
                   f"MATCH (a:{_src} {{id: row.s}}), (b:{_dst} {{id: row.t}}) "
                   f"CREATE (a)-[:{_rel}]->(b)", rows=_rows[_start:_start + 20000])

print("노드:", run_cypher("MATCH (n) RETURN count(n) AS c")[0]["c"],
      "/ 관계:", run_cypher("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"])

In [ ]:
# labels(n)[0]: 노드에 레이블이 여러 개 붙을 수 있어 첫 번째만 본다
for row in run_cypher("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS cnt "
                      "ORDER BY cnt DESC"):
    print(f"  {row['label']:20} {row['cnt']:>6,}")

In [ ]:
# 관계는 방향·타입을 가리지 않고 한 번에 센다
print('관계:', run_cypher('MATCH ()-[r]->() RETURN count(r) AS c')[0]['c'], '건')

## 1-2. 오늘 얹을 논문 69편

### 어떤 글인가
`data/pmc_docs.jsonl` 에 PubMed Central 공개 논문 69편이 있습니다. **JSON Lines** 형식이라 **한 줄이 논문 한 편**이고, 줄마다 아래 여섯 칸을 가집니다. 원문은 영어이고, 우리가 붙이는 설명과 질문은 한국어입니다.

> **`.json` 과 `.jsonl` 의 차이**: `.json` 은 파일 전체가 하나의 JSON 이라 통째로 읽어야 하고, `.jsonl` 은 **줄마다 하나씩 완결된 JSON** 이라 한 줄씩 읽어 처리할 수 있습니다.

| 칸 | 무엇 | 이 단원에서 쓰는 곳 |
|---|---|---|
| `pmcid` | 논문 번호. `PMC13432136` | **노드를 찾는 열쇠.** 검색 결과·인용이 전부 이 값 |
| `title` | 제목 | 2-1 에서 본문과 이어 임베딩 |
| `journal` | 실린 학술지. `PLOS One` | 화면에 보여 줄 때 |
| `year` | 발행 연도. 68편이 2026, 한 편이 2025 | 3-2 에서 조건으로 걸러 본다 |
| `license` | 이용 조건. 69편 모두 `CC BY` | 재배포 조건 확인 |
| `text` | 초록 + 본문 발췌. 3,185~8,248자 | 2-1 에서 임베딩, 교안_02 에서 근거로 주입 |

### `pmcid` 가 이 단원의 열쇠입니다
PubMed Central 이 논문마다 매기는 번호로, `PMC` 뒤에 숫자가 붙습니다. 그 번호가 곧 **원문 주소**가 됩니다(`https://pmc.ncbi.nlm.nih.gov/articles/PMC13432136/`). 세상에 하나뿐인 값이라 **우리가 따로 번호를 만들 필요가 없습니다.** 이런 값을 **자연 키**라고 부릅니다(32일차의 `id` 와 같은 자리입니다).

그래서 이 번호가 단원 내내 따라다닙니다. 2-1 에서 `:Document` 노드를 이 값으로 찾도록 인덱스를 걸고, 3절의 검색 결과도 이 번호로 받고, 교안_02 에서 모델이 답에 다는 인용도 이 번호입니다. **번호 하나로 노트북 안의 결과와 바깥의 원문이 이어집니다.**

코드로 바로 들어가지 말고 어떤 글인지부터 봅니다.

In [ ]:
# 오늘 얹을 논문이 어떤 것인지 먼저 두 편만 펼쳐 봅니다.
import json
from pathlib import Path

DATA_DIR = Path('data') if Path('data').exists() else Path('../data')   # 정답 폴더에서 열어도 돌게
# jsonl 은 한 줄이 논문 한 편이다. 빈 줄은 건너뛰고 줄마다 json 으로 읽는다
papers = [json.loads(line) for line in
          (DATA_DIR / 'pmc_docs.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
print('논문 수:', len(papers))
print('한 편이 가진 칸:', list(papers[0]))

In [ ]:
for paper in papers[:2]:
    print(paper['pmcid'], '|', paper['year'], '|', paper['title'][:58])
    # text 는 초록에 본문 발췌를 이어 붙인 것이라 길다
    print('   길이', len(paper['text']), '자 |', paper['text'][:120].replace('\n', ' '), '...')

### 오늘 만들 모양

논문을 `:Document` 노드로 올리고, 그 논문이 언급한 개체와 `MENTIONS` 로 잇습니다. 그러면 **벡터로 문서를 찾고, 거기서 그래프로 넘어가는** 길이 생깁니다.

<img src="images/문서를_그래프에_얹기.png" alt="Document 노드와 MENTIONS 로 이어지는 지식그래프 개체들" width="900">

그림의 왼쪽이 검색이 데려다주는 곳이고, 거기서 끝나면 그냥 문서 검색입니다. 논문이 이름을 적은 개체에서 출발해 오른쪽 그래프를 따라가는 것이 오늘의 목표입니다. 잇는 열쇠는 이름이 아니라 `id` 입니다(같은 이름이 종류가 다른 노드 둘에 걸리는 일이 있습니다. 5절에서 봅니다).

> **`MENTIONS` 는 약물에만 걸리지 않습니다.** 유전자·질병·증상·약효분류에도 걸리고, 이 코퍼스에서는 **유전자가 가장 많습니다**(588건 중 318건, 69편 중 58편). 그림에는 두 갈래만 그렸습니다.

## 1-3. 임베딩: 지난 교과목에서 배운 것

한 문장은 길이와 무관하게 **768개 숫자**가 되고, 두 문장의 가까움은 **코사인 유사도**로 잽니다. 여기까지는 지난 교과목 그대로입니다. 다만 **모델이 바뀌었습니다.** 지난 교과목의 `ko-sroberta` 대신 오늘은 OpenAI `text-embedding-3-large` 를 씁니다. 차원은 똑같이 768 이지만 **값은 전혀 다릅니다.** **오늘 새로운 것은 이 벡터를 그래프 노드에 담아, 가까운 것을 찾는 일을 데이터베이스에 넘긴다는 것입니다.**

> 3절에서 데이터베이스가 돌려주는 점수는 **같은 벡터를 쓰고도 눈금이 다릅니다.** 아래에서 코사인을 직접 재 견줄 기준을 만들어 둡니다.

In [ ]:
# 임베딩이 어떤 모양으로 나오는지 확인합니다. 오늘 할 일은 이 벡터를 노드에 담는 것입니다.
sample = ['CYP2C19 affects how the body handles clopidogrel.', 'Genetic differences change how a drug is broken down.', 'The weather was warm and the park was crowded.']   # 앞의 둘은 같은 이야기, 세 번째는 딴 이야기
vectors = embed_texts(sample)
# 문장이 길든 짧든 숫자 개수는 같다. 그 '같은 길이'가 있어야 인덱스를 만들 수 있다
print('벡터 개수:', len(vectors), '/ 차원:', len(vectors[0]))

In [ ]:
# 크기(길이)도 재 봅니다. 길이 1 로 정규화돼 나온다는 것을 눈으로 확인합니다
print('벡터 크기:', [round(sum(v * v for v in vec) ** 0.5, 3) for vec in vectors])

문장 셋이 모두 **길이가 같은 768개 숫자**가 됐습니다. 오늘은 이 벡터를 **그래프 노드에 담고**, 가까운 것을 찾는 일을 데이터베이스에 넘깁니다.

### 오늘 쓰는 임베딩 모델의 성질

3절에서 인덱스를 만들 때 **차원**과 **유사도 함수**를 적어 내야 합니다. 그 값이 어디서 오는지가 여기 있습니다.

| | `text-embedding-3-large` (오늘) | `text-embedding-3-small` |
|---|---|---|
| 한 번에 넣을 수 있는 길이 | 8,191 토큰 | 8,191 토큰 |
| 기본 차원 | 3,072 | 1,536 |
| 값 | 100만 토큰당 $0.13 | 100만 토큰당 $0.02 |

- **길이에 상한이 있습니다.** 다만 넘겨도 **에러가 나지 않습니다.** `OpenAIEmbeddings` 가 넘치는 글을 8,191 토큰씩 잘라 따로 임베딩한 뒤 **평균 낸 벡터 하나**를 돌려줍니다. 티가 안 나게 뭉개지는 쪽이라 에러보다 성가십니다. 오늘 논문은 가장 긴 것이 1,953 토큰(중앙값 1,023)이라 통째로 들어갑니다. 긴 문서를 뜻이 살아 있게 나누는 법이 청킹입니다(16일차).
- **차원을 줄여 받을 수 있습니다.** 준비 셀의 `dimensions=768` 이 3,072개 중 **앞 768개만** 받는 설정입니다. 뒤를 버려도 뜻이 크게 상하지 않게 학습돼 있어, 공식 문서는 3-large 를 256차원까지 줄여도 옛 모델 `ada-002` 의 1,536차원보다 낫다고 밝힙니다. 차원이 작을수록 저장 공간도 검색 비용도 줄어듭니다.
- **나오는 벡터는 길이가 1 입니다.** `dimensions` 파라미터는 그 정규화까지 해서 돌려줍니다.
- **다국어 모델입니다.** 한국어 문장과 영어 문장을 같은 공간에 놓기 때문에, 3-2 에서 한국어 질문으로 영어 논문을 찾을 수 있습니다.

3절부터는 가까움을 재는 일을 데이터베이스가 대신합니다. 그 전에 같은 벡터로 코사인 유사도를 직접 재 두면, 뒤에서 데이터베이스가 돌려주는 점수와 견줄 기준이 생깁니다. **이 숫자를 기억해 두세요.** 3절의 검색 점수는 이것과 다른 눈금을 씁니다.

In [ ]:
# 벡터 길이가 1 이라 내적이 곧 코사인 유사도입니다. 세 문장을 한 번에 곱해 서로의 유사도를 봅니다.
import numpy as np

# 어느 번호가 어느 문장인지 먼저 보고 점수를 읽습니다
for i, sentence in enumerate(sample):
    print(f'{i}번: {sentence}')

In [ ]:
similarity = np.array(vectors) @ np.array(vectors).T   # (3, 768) x (768, 3) -> 3x3 유사도 표
print()
print('0번-1번(같은 이야기):', round(float(similarity[0, 1]), 3))
print('0번-2번(딴 이야기)  :', round(float(similarity[0, 2]), 3))

### ✅ 바로 확인 퀴즈 (1-1·1-3)

**1.** 지식그래프와 논문은 같은 사실을 다른 모양으로 담고 있습니다. 논문을 그래프에 얹으려는 이유는?

<details><summary>정답 보기</summary>

그래프는 **조회가 정확하지만 정리된 것만** 알고, 논문은 **최신이지만 찾기가 어렵습니다.** 둘을 한자리에 두면 논문을 뜻으로 찾은 뒤 그 논문이 가리키는 개체를 따라 **그래프가 아는 사실까지** 이어 붙일 수 있습니다.

</details>

**2.** 위에서 잰 유사도는 같은 이야기끼리 0.52, 딴 이야기와 -0.03 이었습니다. 같은 이야기인데 1 에 한참 못 미치는 이유는? 그래서 이 값을 어떻게 읽어야 하나요?

<details><summary>정답 보기</summary>

코사인 유사도는 **두 벡터가 이루는 각**을 재는 값이고, 같은 뜻이라도 표현이 다르면 임베딩 모델은 각을 벌려 놓습니다. 절댓값은 모델마다 다릅니다(다른 모델에서는 같은 두 문장이 0.87 이 나오기도 합니다). 그래서 **절댓값을 기준선으로 삼지 말고 어느 쪽이 더 가까운지(순위)로 읽어야** 합니다. 이 성질은 3절의 검색 점수에서도 그대로 나타납니다.

</details>

---
# 2. 논문을 노드로 올리고 임베딩 붙이기

논문 69편을 `:Document` 노드로 올리고, 각 논문의 임베딩을 노드 속성에 붙입니다.

## 2-1. 임베딩을 노드 속성으로 적재하기

### 왜 필요할까요?
검색할 때마다 논문 69편을 다시 임베딩하면 느리고 비쌉니다. 그래서 **올릴 때 미리 임베딩**해 **노드 속성**으로 저장해 둡니다. 검색 때는 질문만 임베딩하면 되죠.

> **오늘은 논문 한 편을 통째로 벡터 하나에 담습니다.** 실무에서는 문서를 문단이나 절 단위로 잘라 **조각마다** 벡터를 만드는 일이 더 흔합니다. 긴 글을 벡터 하나로 누르면 여러 주제가 뒤섞여 어느 쪽으로도 가깝지 않은 벡터가 되고, 찾은 뒤에도 어느 대목이 근거인지 짚을 수 없기 때문입니다. 우리 논문은 가장 긴 것이 1,100 단어쯤이라 벡터 하나로 담아도 됩니다.

> **768 이라는 숫자도 우리가 고른 것입니다.** 이 모델이 기본으로 내놓는 벡터는 숫자 3,072개인데, 준비 셀이 `dimensions=768` 로 **768개짜리를 달라고 요청**합니다. 짧게 줄여도 뜻이 크게 상하지 않도록 학습된 모델이라 이렇게 쓸 수 있고, 저장 공간과 검색 비용이 그만큼 줄어듭니다. 대신 한 번 정한 차원은 인덱스와 저장해 둔 벡터가 함께 쓰는 값이라 도중에 바꾸지 않습니다.

### 문법: `db.create.setNodeVectorProperty`
임베딩 벡터를 노드에 저장할 때는 **전용 프로시저**를 씁니다. `SET n.emb = [...]` 로 넣은 리스트도 벡터 인덱스가 색인하기는 하지만, 프로시저는 같은 벡터를 **디스크에 더 작게** 담습니다.

정작 조심할 것은 **인덱스를 만들 때 적어 주는 차원**입니다. 저장한 벡터와 그 값이 다르면 검색에서 문제가 됩니다(3-1 에서 봅니다).

### 문법: `UNWIND ... CREATE ... SET n += row`
아래 셀이 논문 69편을 한 줄로 올립니다. 두 조각으로 나눠 읽으면 됩니다.

```cypher
UNWIND $rows AS row CREATE (n:Document) SET n += row
```

- **`UNWIND $rows AS row`**: 파이썬에서 넘긴 **리스트를 행으로 폅니다.** 69개짜리 리스트를 넘기면 뒤의 문장이 69번 돕니다. 32일차에서 파일을 파이썬이 읽어 넘길 때 쓴 그것입니다. 논문마다 쿼리를 보내면 왕복이 69번인데, 이렇게 하면 **한 번**입니다.
- **`SET n += row`**: `row` 는 딕셔너리이고, 그 **키가 그대로 노드 속성 이름이 됩니다.** `pmcid`·`title`·`journal`·`year`·`license`·`text` 여섯 칸을 하나씩 적을 필요가 없습니다.

> **`+=` 와 `=` 는 다릅니다.** `SET n = row` 는 노드의 속성을 **통째로 갈아끼워** 원래 있던 속성이 사라지고, `SET n += row` 는 **있던 것은 두고 딕셔너리의 키만 얹습니다.** 같은 키가 있으면 새 값으로 덮습니다. 노드를 방금 만들었으니 여기서는 결과가 같지만, 이미 있는 노드를 고칠 때는 `=` 를 잘못 쓰면 다른 속성이 날아갑니다.

<img src="images/임베딩을_노드속성으로.png" alt="논문은 올릴 때 한 번 임베딩해 노드 속성으로 저장하고, 검색 때는 질문만 임베딩한다" width="900">

In [ ]:
# 논문 69편을 Document 노드로 올리고, 각 논문의 임베딩을 노드 속성 emb 에 붙입니다.
run_cypher('CREATE INDEX document_pmcid IF NOT EXISTS FOR (n:Document) ON (n.pmcid)')   # 아래 MATCH 가 전수 스캔을 안 하도록
# UNWIND: 넘긴 리스트를 한 줄씩 풀어 69편을 한 번의 왕복으로 올린다(n += row: dict 의 키가 그대로 속성이 된다)
run_cypher('UNWIND $rows AS row CREATE (n:Document) SET n += row', rows=papers)

# 제목과 본문을 이어 임베딩합니다. 제목에만 있는 단어(약 이름 등)을 놓치지 않으려고요
vectors = embed_texts([paper['title'] + ' ' + paper['text'] for paper in papers])

# 벡터는 전용 프로시저로 저장한다. SET 으로 넣은 리스트보다 공간을 덜 쓰는 형태로 담아 준다
run_cypher('''UNWIND $rows AS row
              MATCH (n:Document {pmcid: row.pmcid})
              CALL db.create.setNodeVectorProperty(n, 'emb', row.vec)''',
           rows=[{'pmcid': paper['pmcid'], 'vec': vector}
                 for paper, vector in zip(papers, vectors)])
print('임베딩이 붙은 논문:',
      run_cypher('MATCH (n:Document) WHERE n.emb IS NOT NULL RETURN count(n) AS c')[0]['c'])

이제 69개의 `Document` 노드가 각자 `emb` 속성(768차원 벡터)을 갖고 있습니다. 검색 준비의 절반이 끝났습니다.

### 🖐️ 함께 따라하기: 제목만 임베딩해서 따로 붙이기

무엇을 임베딩하느냐가 검색을 바꿉니다. 위에서는 **제목과 본문을 함께** 임베딩했는데, 이번에는 **제목만** 임베딩해 다른 속성에 담아 둡니다. 3-2 의 따라하기에서 두 결과를 비교합니다.

1. 모든 `Document` 의 `title` 만 `embed_texts` 로 **한 번에** 임베딩하세요.
2. `db.create.setNodeVectorProperty` 로 각 노드의 **`emb_title`** 에 저장하세요(위 셀처럼 `UNWIND` 로 한 번에 보내세요).
3. `emb_title` 을 가진 `Document` 수를 세어 출력하세요.

**확인 기준**: `제목 임베딩이 붙은 논문: 69` 가 나오면 맞습니다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈

**1.** 어떤 학생이 `db.create.setNodeVectorProperty` 대신 `SET n.emb = [...]` 로 벡터를 넣었더니 의미 검색이 한 건도 안 나옵니다. 원인으로 먼저 의심할 것은 무엇인가요?

<details><summary>정답 보기</summary>

**저장 방법이 아니라 차원**입니다. `SET` 으로 넣은 리스트도 벡터 인덱스가 색인하므로 그것만으로는 검색이 비지 않습니다. 검색을 실제로 깨뜨리는 것은 **인덱스에 적어 둔 차원과 넣은 벡터의 차원이 어긋난 것**입니다. 이때는 에러도 나지 않습니다. **차원이 안 맞는 노드가 조용히 색인에서 빠져** 검색 결과만 비거나 모자랍니다. 전용 프로시저를 쓰는 이유(디스크에 알뜰하게 담기)는 검색의 성패와 상관이 없습니다.

</details>

**2.** 논문을 **미리** 임베딩해 저장하는 이유는?

<details><summary>정답 보기</summary>

검색 때마다 논문 전체를 다시 임베딩하면 느리고 비용이 듭니다. 미리 저장해 두면 **검색 때는 질문만** 임베딩하면 됩니다.

</details>

---
# 3. 벡터 인덱스와 의미 검색

임베딩을 붙여 두면 가까운 문서를 찾을 수는 있습니다. 다만 문서를 하나씩 다 재야 하고, `SEARCH` 절도 쓸 수 없습니다. 그 벡터를 **인덱스에 걸어** 두는 것이 3절의 일입니다.

- **3-1** 벡터 인덱스를 만듭니다(차원과 유사도 함수를 적어 주는 것이 핵심).
- **3-2** `SEARCH` 절로 의미 검색하고, `LIMIT` 과 조건의 순서를 봅니다.

## 3-1. 벡터 인덱스 만들기: 차원과 유사도 함수 정하기

### 왜 필요할까요?
저장만 해선 검색이 느립니다. 질문 벡터와 **모든** 문서 벡터를 일일이 비교해야 하니까요. **벡터 인덱스**는 가까운 벡터를 빠르게 찾도록 미리 구조를 잡아 둡니다(문서가 많을수록 효과가 큽니다).

> 다만 **69편에서는 그 차이가 밀리초 단위입니다.** 아래에서 직접 재 보는데, 인덱스가 제 값을 하는 것은 문서가 수만·수십만이 됐을 때입니다. 지금 만드는 또 다른 이유는 `SEARCH` 절이 **인덱스를 요구하기 때문**입니다.

> **그리고 이 인덱스는 '가장 가까운 것'을 보장하지 않습니다.** 전수 비교 대신 미리 잡아 둔 이웃 구조를 따라가며 후보만 살피기 때문에, 진짜 상위 k 편과 어긋날 수 있습니다. 이런 방식을 **근사(approximate) 최근접 이웃 검색**이라고 부르고, Neo4j 벡터 인덱스의 구조는 15·17일차에서 본 **HNSW** 입니다. 69편처럼 작은 데이터에서는 대개 전수 비교와 같은 답이 나오지만, 문서가 많아지면 갈릴 수 있습니다. 그래서 검색 품질을 따질 때는 "인덱스가 준 답" 과 "진짜 정답" 을 구분해서 봅니다.

### 문법: `CREATE VECTOR INDEX`
- **`vector.dimensions`: 768**. 저장한 임베딩 차원과 **똑같이**. 어긋나면 그 벡터는 조용히 색인에서 빠지고, 검색할 때 다른 차원의 질문 벡터를 넘기면 에러가 납니다.
- **`vector.similarity_function`: 'cosine'**. 가까움을 **코사인으로 재고**, 그 값을 0~1 로 눌러 점수로 돌려줍니다(어떻게 누르는지는 3-2 에서 봅니다). Neo4j 가 받는 함수는 `cosine`(두 벡터가 이루는 각)과 `euclidean`(거리) 둘뿐이고, **임베딩 모델이 권장하는 것**을 고릅니다(OpenAI 임베딩은 코사인).

> 몇 건을 받을지(**top_k**)는 뒤에 붙일 일에 맞춰 정합니다. 근거로 모델에 넘길 거라면 3~5건이 흔합니다. 적으면 답의 근거가 모자라고, 많으면 관련 없는 문서까지 섞입니다.

> 인덱스를 만든 뒤에는 `db.awaitIndexes()` 로 **준비될 때까지 기다립니다**. 바로 검색하면 아직 인덱스가 안 서 있을 수 있습니다.

<img src="images/벡터인덱스_왜필요한가.png" alt="인덱스 없이 전수 비교할 때와 벡터 인덱스로 후보만 비교할 때의 차이" width="900">

인덱스는 만들 때 한 번 품이 들고, 검색할 때마다 그 품을 돌려받습니다. **전수 비교는 문서 수에 비례해 늘고 인덱스는 그렇지 않습니다.** 먼저 인덱스 없이 재 봅니다. `vector.similarity.cosine(a, b)` 이 두 벡터의 가까움을 그 자리에서 재 줍니다.

In [ ]:
# 인덱스 없이 재는 법: 논문을 하나씩 다 재서 정렬한다(vector.similarity.cosine 은 3-2 의 검색 점수와 같은 눈금이다)
import time

preview_vec = embed_query('어떤 유전자가 약물 대사에 관여하나요?')
BRUTE = '''MATCH (n:Document)
           RETURN n.pmcid AS pmcid,
                  round(vector.similarity.cosine(n.emb, $q), 3) AS score
           ORDER BY score DESC LIMIT 3'''
for row in run_cypher(BRUTE, q=preview_vec):
    print(' ', row['pmcid'], row['score'])

In [ ]:
def measure_ms(cypher, vec):
    """같은 질의를 세 번 재서 가장 짧은 시간(ms). 한 번만 재면 그때의 잡음이 그대로 값이 된다."""
    times = []
    for _ in range(3):
        start = time.perf_counter()
        run_cypher(cypher, q=vec)
        times.append((time.perf_counter() - start) * 1000)
    return min(times)


brute_ms = measure_ms(BRUTE, preview_vec)
print(f'전수 비교: {brute_ms:.1f} ms')

인덱스가 없어도 답은 나옵니다. **이 목록과 시간을 기억해 두세요.** 3-2 에서 인덱스로 같은 질문을 검색해 둘 다 견줍니다. `vector.similarity.cosine` 이 돌려주는 값은 3-2 의 검색 점수와 **같은 눈금**이라 숫자를 그대로 맞대 볼 수 있습니다.

In [ ]:
# 벡터 인덱스를 만듭니다. 차원은 임베딩과 같은 768, 유사도 함수는 모델이 권하는 코사인으로 맞춥니다.
run_cypher('''CREATE VECTOR INDEX doc_vec IF NOT EXISTS FOR (n:Document) ON n.emb
              OPTIONS {indexConfig: {`vector.dimensions`: 768,
                                     `vector.similarity_function`: 'cosine'}}''')
run_cypher('CALL db.awaitIndexes()')          # 인덱스가 준비될 때까지 기다립니다
print('벡터 인덱스 doc_vec 생성 완료')

인덱스가 어떤 값으로 섰는지는 `SHOW VECTOR INDEXES` 로 언제든 볼 수 있습니다. 우리가 적은 차원·유사도 함수 옆에 `vector.hnsw.*` 옵션이 기본값으로 채워져 있습니다. 이 인덱스가 HNSW 구조라는 뜻입니다.

In [ ]:
# 인덱스가 어떤 값으로 섰는지 확인한다. state 가 ONLINE 이어야 검색에 쓰인다
for row in run_cypher('''SHOW VECTOR INDEXES YIELD name, state, properties, options
                         RETURN name, state, properties, options.indexConfig AS config'''):
    print(row['name'], row['state'], row['properties'])
    # 차원·유사도 함수는 우리가 적은 값이고, hnsw·양자화 옵션은 기본값으로 채워졌다
    for key, value in sorted(row['config'].items()):
        print(f'   {key:38} {value}')

## 3-2. 의미 검색: `SEARCH` 절

### 문법: `SEARCH ... IN (VECTOR INDEX ...)`
찾을 노드를 `MATCH` 로 잡고, 그 노드를 **`SEARCH ... IN (VECTOR INDEX ...)`** 로 벡터 인덱스에 넘깁니다.

```
MATCH (n:Document)
  SEARCH n IN (VECTOR INDEX 인덱스이름 FOR 질문벡터 LIMIT 몇개) SCORE AS score
RETURN n.pmcid, score
```

`LIMIT` 이 몇 건을 받을지(top_k)이고, `SCORE AS score` 로 그 점수에 이름을 붙여 받습니다.

> **인덱스를 타느냐가 3-1 과 다른 점입니다.** `vector.similarity.cosine(a, b)` 는 **두 벡터를 그 자리에서 재는 함수**라 인덱스와 무관합니다. 그래서 3-1 처럼 `MATCH (n:Document)` 로 **69편을 전부 훑어** 하나씩 재고 정렬해야 합니다. `SEARCH` 는 **인덱스에 물어** 가까운 쪽만 훑습니다. 눈금이 같아 두 점수를 그대로 맞대 볼 수 있습니다.

> **`LIMIT` 은 거르기 전에 자릅니다.** 인덱스에서 먼저 상위 k 건을 뽑고, 뒤에 `WHERE` 를 붙이면 **그 k 건 안에서만** 거릅니다. 그래서 조건에 맞는 문서가 다른 데 더 있어도 채워 주지 않습니다. 잠시 뒤 직접 확인합니다.

> **벡터 인덱스로 검색할 때는 이 `SEARCH` 절을 씁니다.** 예전에는 `CALL db.index.vector.queryNodes(...)` 프로시저를 불렀습니다. 지금 버전은 그 프로시저를 **폐기 예정으로 표시하고 대신 `SEARCH` 를 쓰라고 안내합니다**(`SHOW PROCEDURES` 의 `deprecatedBy` 칸에 `SEARCH` 라고 적혀 있습니다). 결과는 같지만 새로 쓰는 코드는 `SEARCH` 로 씁니다.

> **다만 `SEARCH` 가 받는 인덱스는 벡터 인덱스뿐입니다.** 4절에서 만들 전문 인덱스는 이 절로 부를 수 없고 프로시저를 그대로 씁니다. 둘을 섞지 않도록 여기서 짝을 지어 둡니다.

| 무엇을 검색하나 | 어떤 인덱스 | 부르는 법 |
|---|---|---|
| 뜻이 가까운 문서 | 벡터 | `MATCH (n:Document) SEARCH n IN (VECTOR INDEX ...)` |
| 같은 것을 인덱스 없이 | - | `vector.similarity.cosine(n.emb, $q)` 로 전부 재서 정렬 (3-1) |
| 키워드가 든 문서 | 전문 | `CALL db.index.fulltext.queryNodes('이름', '검색어')` (4-1) |

In [ ]:
# 질문을 임베딩해서, 벡터 인덱스로 '뜻이 가까운' 논문 상위 3편을 찾습니다.
question = '어떤 유전자가 약물 대사에 관여하나요?'
q_vec = embed_query(question)   # 문서와 같은 모델·같은 768차원으로 질문도 임베딩한다
SEARCH = '''MATCH (n:Document)
              SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 3) SCORE AS score
            RETURN n.pmcid AS pmcid, n.title AS title, round(score, 3) AS score
            ORDER BY score DESC'''   # LIMIT 3 이 top_k, 즉 몇 편을 받을지다
hits = run_cypher(SEARCH, q=q_vec)   # 5-3 에서 이 1위 논문으로 그래프를 건너간다
for hit in hits:
    print(hit['pmcid'], '| 유사도', hit['score'], '|', hit['title'][:56])

In [ ]:
# 3-1 의 전수 비교와 같은 방법으로 재서 나란히 놓는다
print(f'전수 비교 {brute_ms:.1f} ms  ->  벡터 인덱스 {measure_ms(SEARCH, q_vec):.1f} ms')

질문은 한국어인데 논문은 영어입니다. 그런데도 약물유전체 논문이 올라옵니다. 1-3 에서 본 **다국어 모델**의 성질이 여기서 그대로 나타난 것입니다.

> **3-1 에서 인덱스 없이 재 본 목록과 견줘 보세요.** 순서도 점수도 같고, 걸린 시간만 줄었습니다. 69편에서는 근사 검색이 전수 비교와 **같은 답**을 냈다는 뜻입니다.

> **점수의 눈금이 1-3 과 다릅니다.** Neo4j 는 코사인을 그대로 주지 않고 **`(1+코사인)/2`** 로 0~1 구간에 옮겨 돌려줍니다(1-3 의 코사인 0.52 는 이 눈금에서 0.76). 옮기면서 폭이 절반으로 눌려 값이 좁은 구간에 몰립니다. 1-3 은 영어 문장 둘을 잰 값이고 여기는 한국어 질문과 논문을 잰 값이라 **크기를 맞대지 말고**, **순위로 읽으세요.**

### `LIMIT` 뒤에 조건을 붙이면
위에서 말한 자르는 순서를 눈으로 봅니다. 같은 검색에 "제목에 `Pharmacogenomic` 이 든 것만" 이라는 조건을 붙이고, 그 조건에 맞는 논문이 원래 몇 편인지도 따로 세어 나란히 놓습니다.

In [ ]:
# 같은 검색에 조건을 하나 붙여 봅니다. LIMIT 5 로 다섯 편을 받은 뒤 제목으로 거른다
kept = run_cypher('''MATCH (n:Document)
                       SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 5) SCORE AS score
                     WHERE n.title CONTAINS 'Pharmacogenomic'
                     RETURN n.pmcid AS pmcid''', q=q_vec)
# 조건에 맞는 논문이 원래 몇 편인지는 검색을 거치지 않고 따로 센다
whole = run_cypher('''MATCH (n:Document) WHERE n.title CONTAINS 'Pharmacogenomic'
                      RETURN count(n) AS c''')[0]['c']
print('LIMIT 5 로 받아 거른 뒤 :', len(kept), '편')
print('조건에 맞는 논문 전체  :', whole, '편')

다섯 편을 요구했는데 걸러 낸 뒤에 남은 것은 그보다 적고, 조건에 맞는 논문은 그보다도 더 있습니다. **엔진이 뒤를 채워 주지 않는다는 뜻입니다.** `SEARCH` 가 먼저 다섯 편을 자르고, `WHERE` 는 그 다섯 편 안에서만 거르기 때문입니다.

> 대처는 둘이고, 부르는 이름이 있습니다.
> - **사후 필터(post-filter)**: `LIMIT` 을 넉넉히 잡아 받은 뒤 거르고 원하는 만큼 자릅니다(다음 시간에 커뮤니티로 좁힐 때 이 순서를 씁니다). 방금 본 것이 이쪽입니다.
> - **사전 필터(pre-filter)**: 조건을 `SEARCH` 안에 넣습니다. 그러면 **조건에 맞는 벡터만 후보가 되고**, 그중에서 가까운 순으로 `LIMIT` 만큼 찾습니다. 그러려면 인덱스를 만들 때 거를 속성을 `WITH [n.year]` 처럼 함께 저장해 두어야 하고, 조건은 **한 값(`=`·`IN`)이나 범위(`>=`·`<=`)** 를 `AND` 로만 이을 수 있습니다(`CONTAINS`·`OR`·`<>` 는 안 됩니다). 저장해 두지 않은 속성을 넣으면 에러가 납니다.

In [ ]:
# 거를 속성을 인덱스에 함께 저장해 둔다: WITH [n.year]. 이렇게 등록한 속성만 SEARCH 안의 WHERE 에 쓸 수 있다
run_cypher('''CREATE VECTOR INDEX doc_vec_year IF NOT EXISTS FOR (n:Document) ON n.emb WITH [n.year]
              OPTIONS {indexConfig: {`vector.dimensions`: 768,
                                     `vector.similarity_function`: 'cosine'}}''')
run_cypher('CALL db.awaitIndexes()')
# 사후 필터: 상위 3편을 먼저 자르고 그 안에서 거른다. 2025년 논문은 한 편뿐이라 살아남을 자리가 없다
behind = run_cypher('''MATCH (n:Document)
                         SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 3) SCORE AS score
                       WHERE n.year = '2025'
                       RETURN n.pmcid AS pmcid''', q=q_vec)
# 사전 필터: 조건이 SEARCH 안에 있으면 조건에 맞는 벡터만 후보로 두고 그중 상위 3편을 찾는다
inside = run_cypher('''MATCH (n:Document)
                         SEARCH n IN (VECTOR INDEX doc_vec_year FOR $q WHERE n.year = '2025' LIMIT 3) SCORE AS score
                       RETURN n.pmcid AS pmcid''', q=q_vec)
print('사후 필터(자르고 거르기):', len(behind), '편')

In [ ]:
print('사전 필터(거르며 찾기)  :', len(inside), '편', [row['pmcid'] for row in inside])

사후 필터는 0편, 사전 필터는 1편입니다. 2025년 논문이 한 편뿐이라 전체 상위 3편 안에는 못 들었지만, 조건에 맞는 것끼리 겨루면 그 한 편이 바로 1위입니다. 인덱스가 하는 일은 여전히 **빨리 찾는 것**이고, 달라진 것은 **거르는 시점**입니다. 거를 속성이 정해져 있으면 사전 필터가 정확하고, `CONTAINS` 처럼 사전 필터가 못 받는 조건이면 넉넉히 받아 뒤에서 자릅니다.

### 🖐️ 함께 따라하기: 무엇을 임베딩했느냐로 결과가 갈리는지 보기

2-1 의 따라하기에서 **`emb_title`**(제목만 임베딩)을 만들어 두었습니다. 같은 질문을 **제목+본문 임베딩**과 **제목만 임베딩**에 각각 걸어 무엇이 올라오는지 비교합니다. 벡터 인덱스는 속성 하나만 가리키므로, 그러려면 `emb_title` 용 인덱스를 하나 더 만들어야 합니다.

1. `Document` 의 `emb_title` 에 **768차원·코사인** 벡터 인덱스 **`doc_title_vec`** 를 만들고 `db.awaitIndexes()` 로 기다리세요.
2. 질문 **'우울증과 관련된 유전자는 무엇인가요?'** 로 `doc_vec`(제목+본문)과 `doc_title_vec`(제목만)에서 각각 **상위 3편**의 `pmcid` 를 받으세요.
3. 두 목록을 나란히 출력하세요.

**확인 기준**: 두 쪽 모두 1위는 우울증과 비만의 유전 연관을 다룬 `PMC13493084` 입니다. 그런데 **2·3위가 다릅니다.** 인덱스 설정은 둘이 똑같으니(768차원·코사인) 이 차이는 **무엇을 임베딩했느냐**에서 온 것입니다. 제목에는 본문에 있는 단어가 대부분 없어서, 제목만 임베딩하면 논문이 실제로 무엇을 말했는지가 벡터에 덜 담깁니다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈 (3-1·3-2)

**1.** 벡터 인덱스를 만들 때 `vector.dimensions` 를 768로 정하는 이유는?

<details><summary>정답 보기</summary>

저장한 임베딩이 **768차원**이기 때문입니다. 어긋나면 자리에 따라 증상이 갈립니다. **질문 벡터**의 차원이 다르면 검색에서 에러가 나고, **저장한 벡터**의 차원이 다르면 에러 없이 그 노드가 색인에서 빠져 결과가 조용히 모자랍니다.

</details>

**2.** 이 절의 검색 점수는 상위 세 편이 **0.731 · 0.708 · 0.706** 입니다. 인덱스에 유사도 함수를 `cosine` 으로 적어 냈는데 왜 코사인값이 그대로 나오지 않을까요? 그래서 이 점수를 어떻게 읽어야 하나요?

<details><summary>정답 보기</summary>

Neo4j 가 돌려주는 점수는 코사인값 그대로가 아니라 **`(1+코사인)/2`** 로 0~1 에 옮긴 값이기 때문입니다. 코사인 0.52 는 점수 0.76 이 되고, 코사인 폭은 절반으로 눌려 값이 좁은 구간에 몰립니다.

그래서 절댓값을 절대 기준으로 쓰지 않고 **순위(어느 문서가 더 가까운가)** 로 읽습니다. 점수 크기는 눌러 맞춘 눈금 위의 값이고, 모델·문장에 따라서도 달라집니다.

</details>

---
# 4. 전문 인덱스: 키워드로 찾기

의미 검색 말고 **글자로 찾는** 길을 하나 더 냅니다. 두 길이 무엇을 서로 놓치는지가 이 절의 요지입니다.

- **4-1** 전문(full-text) 인덱스를 만들어 키워드로 찾습니다.
- **4-2** 의미 검색과 글자 검색이 갈리는 자리를 봅니다.
- **4-3** 하이픈이 든 기호가 왜 따옴표를 요구하는지 봅니다.

## 4-1. 전문 인덱스 만들고 키워드로 찾기

### 왜 필요할까요?
의미 검색이 항상 정답은 아닙니다. **정확한 유전자 기호·약 이름·오류 코드**를 찾을 땐 그 **글자가 든 문서**를 곧장 찾는 게 낫습니다. 이런 **글자 검색**을 해 주는 것이 **전문(full-text) 인덱스**입니다. 이 교안에서는 앞으로 이 방식을 **전문 검색**이라고 부릅니다(키워드 검색·글자 검색이 다 같은 말입니다).

### 문법: `CREATE FULLTEXT INDEX` · `db.index.fulltext.queryNodes`
- 만들기: `CREATE FULLTEXT INDEX 이름 FOR (n:Document) ON EACH [n.title, n.text]`. 어떤 텍스트 속성을 단어 단위로 색인할지 지정합니다.
- 검색: `db.index.fulltext.queryNodes('이름', '검색어')` 로 그 **키워드가 든 문서**를 점수와 함께 받습니다.
- 검색어는 Lucene 문법입니다. 키워드를 `AND`·`OR` 로 묶고, `title:CYP2C19` 처럼 속성 이름을 앞에 붙이면 그 속성에서만 찾습니다. 대소문자는 가리지 않습니다.

> **여기는 3-2 의 `SEARCH` 절이 아니라 프로시저입니다.** `SEARCH` 는 벡터 인덱스만 받아서 `SEARCH n IN (FULLTEXT INDEX ...)` 라고 쓰면 문법 오류가 납니다(`expected 'VECTOR INDEX'`). 전문 인덱스는 프로시저로 부릅니다.

> **여기서 인덱스는 '빠르게' 만 하는 것이 아닙니다.** 3-1 에서 본 것처럼 벡터 검색은 인덱스가 없어도 `vector.similarity.cosine` 한 줄로 같은 답을 냈습니다(느릴 뿐입니다). 전문 검색은 그런 한 줄이 없습니다. `db.index.fulltext.queryNodes` 는 **인덱스가 있어야만** 부를 수 있고, 없으면 그 자리에서 에러가 납니다.

> 물론 `CONTAINS`·`split()`·정규식 같은 **문자열 함수로 흉내는 낼 수 있습니다.** 다만 그때는 단어로 쪼개는 규칙, 대소문자 처리, `AND`·`OR` 같은 결합, 점수 매기기를 **직접 짜야** 합니다. 전문 인덱스는 그 네 가지를 미리 갖춰 둔 것이고(쪼개는 규칙을 analyzer 라고 합니다. 4-3 에서 봅니다), 속도는 그 위에 얹힌 이득입니다.

In [ ]:
# 전문 인덱스는 '그 키워드가 든' 문서를 찾습니다(뜻이 아니라 글자 일치). ON EACH 에 적은 두 속성의 글자를 단어로 쪼개 색인합니다.
run_cypher('''CREATE FULLTEXT INDEX doc_ft IF NOT EXISTS
              FOR (n:Document) ON EACH [n.title, n.text]''')
run_cypher('CALL db.awaitIndexes()')   # 색인이 다 설 때까지 기다린다

# 전문 검색은 아직 프로시저를 부른다. score 는 키워드가 얼마나 잘 맞았는지의 점수다
term_hits = run_cypher('''CALL db.index.fulltext.queryNodes('doc_ft', $term)
                          YIELD node, score
                          RETURN node.pmcid AS pmcid ORDER BY score DESC''', term='CYP2C19')
print('CYP2C19', '로 전문 검색에 걸린 논문:', len(term_hits), '편')

In [ ]:
print([hit['pmcid'] for hit in term_hits])

`CYP2C19` 는 유전자 기호입니다. 글자가 그대로 적힌 논문만 정확히 걸립니다.

> 이 기호는 하이픈이 없어 **단어 하나로 그대로** 색인됩니다. 그래서 이 경우에는 걸린 편수와 원문에 실제로 적힌 편수가 같습니다. 늘 그런 것은 아닙니다. 4-3 에서 둘이 어긋나는 경우를 봅니다.

검색어 문법도 한 번 써 봅니다. 같은 기호를 다른 단어와 묶거나 제목으로 한정하면 편수가 달라집니다.

In [ ]:
# 검색어는 Lucene 문법이다. 키워드를 AND/OR 로 묶고, 속성이름: 을 앞에 붙이면 그 속성에서만 찾는다
for term in ['CYP2C19 AND warfarin', 'CYP2C19 OR warfarin', 'title:CYP2C19']:
    rows = run_cypher('''CALL db.index.fulltext.queryNodes('doc_ft', $t) YIELD node
                         RETURN count(node) AS c''', t=term)
    print(f'  {term:22} -> {rows[0]["c"]}편')

## 4-2. 의미 검색과 갈리는 자리

같은 기호를 **의미 검색에도 넣어** 나란히 놓습니다. 두 방식이 서로 무엇을 놓치는지가 여기서 보입니다.

<img src="images/의미검색_vs_전문검색.png" alt="의미 검색은 뜻으로 찾고 전문 검색은 글자로 찾는다. 서로 놓치는 것이 다르다" width="900">

In [ ]:
# 의미 검색은 글자가 맞는지와 무관하게 LIMIT 만큼 채워 온다. 여기서는 5편
vec_for_term = run_cypher('''MATCH (n:Document)
                               SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 5) SCORE AS score
                             RETURN n.pmcid AS pmcid ORDER BY score DESC''',
                          q=embed_query('CYP2C19'))
term_ids = {hit['pmcid'] for hit in term_hits}   # 겹치는지 빠르게 보려고 집합으로 둔다
print('전문 검색(키워드):', [hit['pmcid'] for hit in term_hits])
print('의미 검색(벡터):', [hit['pmcid'] for hit in vec_for_term])

In [ ]:
print('벡터 상위 5편 중 그 기호가 실제로 적힌 논문:',
      sum(1 for hit in vec_for_term if hit['pmcid'] in term_ids), '편')

의미 검색은 **약물유전체 이야기를 하는 논문**을 올립니다. 그 기호가 실제로 적혀 있는지는 따지지 않습니다. "이 유전자가 나오는 논문을 빠짐없이 달라"는 요구라면 **전문 검색이 맞습니다.**

반대 방향도 봅니다. 한국어 질문을 두 방식에 똑같이 넣으면 어떻게 될까요?

In [ ]:
# 반대로, 한국어를 두 방식에 넣어 봅니다. 전문 검색에는 키워드 '유전자'를, 의미 검색에는 질문 문장 전체를 넣습니다.
korean_ft = run_cypher('''CALL db.index.fulltext.queryNodes('doc_ft', $term)
                          YIELD node RETURN node.pmcid AS pmcid''', term='유전자')
korean_vec = run_cypher('''MATCH (n:Document)
                             SEARCH n IN (VECTOR INDEX doc_vec FOR $q LIMIT 3) SCORE AS score
                           RETURN n.pmcid AS pmcid ORDER BY score DESC''',
                        q=embed_query(question))
# 두 결과를 한 셀에서 나란히 본다. 전문이 0건이어도 의미 검색은 답한다
print("전문 검색 '유전자'  :", f'{len(korean_ft)}건')
print('의미 검색(같은 물음):', [hit['pmcid'] for hit in korean_vec])

전문 검색은 **0건**입니다. 원문이 영어라 '유전자'라는 글자가 어디에도 없기 때문입니다. 같은 물음을 문장째 넣은 의미 검색은 관련 논문을 찾아냅니다. **글자가 없어도 뜻이 통하면 찾는 것**이 벡터 검색입니다.

> 다만 이 시연이 "벡터가 더 좋다" 를 증명한 것은 아닙니다. 영어 말뭉치에 한국어 키워드를 넣었으니 전문 검색이 0건인 것은 처음부터 정해진 결과입니다. 앞의 `CYP2C19` 비교가 반대 방향을 보여 줍니다. 그 기호를 빠짐없이 모으는 일에서는 전문 검색이 낫습니다. 요지는 우열이 아니라 **어느 쪽이 무엇을 놓치는가** 입니다.

## 4-3. 정확한 기호라고 늘 정확히 걸리지는 않습니다

쪼개기는 **두 곳에서** 일어납니다. **문서를 색인할 때 한 번, 검색어를 받을 때 한 번**, 같은 규칙이 양쪽에 적용됩니다. 그 규칙을 **analyzer** 라고 부르고, 기본값 `standard-no-stop-words` 는 소문자로 눌러 담으며 **공백과 문장부호에서 단어를 끊습니다.** 하이픈도 그 문장부호에 듭니다.

- **저장할 때**: 원문의 `HLA-B` 는 인덱스에 `hla` 와 `b` **두 단어로** 들어갑니다. `hla-b` 라는 한 단어는 인덱스에 없습니다.
- **검색할 때**: 검색어 `HLA-B` 도 똑같이 `hla` 와 `b` 로 쪼개집니다. 기본 결합이 `OR` 이라 **둘 중 하나만 있어도** 걸립니다.

그래서 기호를 그냥 넣으면 엉뚱한 논문까지 걸려 나옵니다. 직접 세어 확인합니다.

> analyzer 는 인덱스를 만들 때 `OPTIONS {indexConfig: {`fulltext.analyzer`: '...'}}` 로 바꿀 수 있고, 고를 수 있는 목록은 `db.index.fulltext.listAvailableAnalyzers()` 가 보여 줍니다. 오늘은 기본값으로 갑니다.

<img src="images/전문인덱스_토큰화.png" alt="HLA-B 를 그냥 넣을 때와 따옴표로 묶을 때 전문 인덱스가 찾는 논문 수의 차이" width="900">

In [ ]:
# 정확한 기호를 찾을 때도 조심할 데가 있습니다. 하이픈이 든 유전자 기호를 그대로 넣어 봅니다.
def fulltext(term):
    """전문 인덱스로 찾은 논문의 pmcid 목록(점수 내림차순)."""
    rows = run_cypher('''CALL db.index.fulltext.queryNodes('doc_ft', $t) YIELD node, score
                         RETURN node.pmcid AS pmcid ORDER BY score DESC''', t=term)
    return [row['pmcid'] for row in rows]


hyphen_term = 'HLA-B'   # 하이픈이 든 유전자 기호
# 원문에 그 글자가 정말로 있는 논문을 파이썬으로 직접 세어 답을 맞춰 봅니다
truth = [paper['pmcid'] for paper in papers
         if hyphen_term in paper['title'] + ' ' + paper['text']]
print('원문에 실제로 있는 논문 :', len(truth), '편')
print('그냥 넣으면            :', len(fulltext(hyphen_term)), '편')
print('따옴표로 묶으면        :', len(fulltext('"' + hyphen_term + '"')), '편')
print('HLA-A 를 그냥 넣으면   :', len(fulltext('HLA-A')), '편')

In [ ]:
# 쪼갠 단어를 하나씩 넣어 봅니다. 앞 단어만으로도 위 truth 와 같은 논문이 나오면 저장할 때도 쪼개진 것입니다
head, tail = hyphen_term.split('-')
print(f'{head} 만 넣으면          :', len(fulltext(head)), '편 · truth 와 같은가:',
      set(fulltext(head)) == set(truth))

In [ ]:
print(f'{tail} 만 넣으면            :', len(fulltext(tail)), '편')

원문에 그 기호가 실제로 있는 논문은 3편인데, 그냥 넣으면 18편이 걸립니다. 쪼갠 뒤 단어 `B` 하나만 넣어도 똑같이 18편입니다. **검색어를 쪼갠 뒤 `OR` 로 합친 결과**라는 뜻입니다. `HLA-A` 가 **69편 전부**인 것도 같은 까닭입니다(`a` 한 글자가 아무 논문에나 있으니까요).

> **앞 단어 `HLA` 만 넣어도 원문에 `HLA-B` 가 있는 그 3편이 그대로 나옵니다.** 그 세 논문에는 홀로 선 `HLA` 가 한 번도 없고 `HLA-B` 만 있습니다. 그런데도 걸린다는 것은 **저장할 때도 쪼개져** 인덱스에 `hla` 라는 단어로 들어가 있다는 증거입니다.

> **따옴표로 묶으면** 3편으로 맞습니다. 다만 따옴표가 쪼개기를 끄는 것은 아닙니다. 여전히 `hla` 와 `b` 로 쪼갠 뒤, 그 둘이 **그 순서로 바로 붙어 있는** 문서만 고르는 **구문 질의**가 됩니다.

> 그래서 하이픈이냐 공백이냐는 쪼갠 뒤에는 구별이 사라집니다. `HLA-B`·`HLA B` 를 그냥 넣으면 둘 다 18편이고, `"HLA-B"`·`"HLA B"` 로 묶으면 둘 다 같은 3편입니다. **따옴표는 기호를 원문 그대로 찾아 주는 장치가 아닙니다.** 두 단어가 붙어 있기만 하면 되니 원문이 `HLA, B` 여도 함께 걸립니다. 이 데이터에서 3편이 정확히 맞은 것은 그런 표기가 없었기 때문입니다. 그래도 기호에 하이픈·점·슬래시가 있으면 **따옴표로 묶는 습관**이 훨씬 낫습니다. 묶지 않으면 단어 하나만 겹쳐도 걸려 나오니까요.

### 의미 검색과 전문 검색: 정리

| | 의미 검색(벡터 인덱스) | 전문 검색(full-text 인덱스) |
|---|---|---|
| 무엇으로 찾나 | 문장의 **뜻**(임베딩) | 문서에 든 **글자** |
| 표현이 달라도 | 찾음(동의어·다른 말투·다른 언어) | 못 찾음(그 글자가 있어야) |
| 정확한 기호·코드 | 애매할 수 있음 | 정확히 찾음(하이픈이 있으면 따옴표로 묶어야) |
| 점수의 눈금 | 0~1(`(1+코사인)/2`) | 상한이 없다(드문 단어일수록, 자주 나올수록 커진다) |
| 준비물 | 임베딩 + 벡터 인덱스 | 전문 인덱스만 |

두 점수는 눈금이 아예 다릅니다. 이 데이터에서 벡터 점수는 0.6~0.7 언저리에 몰리는데, 전문 점수는 0.01 도 안 되는 것부터 3.6 을 넘는 것까지 벌어집니다. 그러니 **두 점수를 더하거나 크기로 견주면 안 됩니다.** 섞어 쓰려면 각각을 따로 순위로 바꾼 뒤 합쳐야 합니다.

둘은 경쟁이 아니라 **역할이 다릅니다**. 실무에서는 대개 둘 다 만들어 두고 질문에 따라 고르거나 섞습니다.

### 🖐️ 함께 따라하기: 다른 기호로 두 방식 비교하기

이번엔 다른 유전자 기호 **`CYP3A4`**(약물 상호작용에서 자주 나오는 대사 효소)로 같은 비교를 해 봅니다.

1. `doc_ft` 로 **`'CYP3A4'`** 를 검색해 나온 `pmcid` 목록을 점수 내림차순으로 **`ft_ids`** 에 담으세요(5-3 에서 다시 씁니다).
2. 같은 기호를 임베딩해 `doc_vec` 에서 **상위 5편**을 받으세요.
3. 두 목록과, **벡터 상위 5편 중 그 기호가 실제로 적힌 논문 수**를 출력하세요.
4. 이번엔 하이픈이 든 기호 **`COX-2`**(염증에 관여하는 효소)로 4-3 을 연습합니다. 위 셀에서 만든 `fulltext` 로 **따옴표 없이** 한 번, **따옴표로 묶어** 한 번 검색하고, 원문에 그 글자가 정말 있는 논문 수도 `papers` 에서 파이썬으로 직접 세어 셋을 나란히 출력하세요.

**확인 기준**: 전문 검색은 **6편**이 걸립니다. 벡터 상위 5편 중 그 기호가 실제로 적힌 논문은 그보다 적습니다. `COX-2` 는 원문에 있는 것이 **2편**인데 그냥 넣으면 **46편**이 걸리고, 따옴표로 묶으면 **2편**으로 맞습니다. `HLA-B` 보다 격차가 큰 이유는 `2` 가 논문마다 흔한 단어이기 때문입니다.

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈 (4-1·4-2·4-3)

**1.** "약을 분해하는 효소"를 "약물 대사 효소"로 바꿔 물어도 찾아 주는 검색은 둘 중 무엇인가요?

<details><summary>정답 보기</summary>

**의미 검색(벡터 인덱스)** 입니다. 뜻으로 찾기 때문에 단어가 달라도 찾습니다. 전문 검색은 그 글자가 든 문서만 찾습니다.

</details>

**2.** 특정 유전자 `VKORC1` 이 언급된 논문을 **빠짐없이** 모으고 싶습니다. 어느 검색이 나을까요?

<details><summary>정답 보기</summary>

**전문 검색(키워드 검색)** 입니다. 정확한 기호·이름은 그 글자가 든 문서를 곧장 집는 쪽이 정확합니다. 의미 검색은 비슷해 보이는 다른 유전자 이야기를 섞을 수 있고, 빠짐없이 모았는지도 보장하지 않습니다.

</details>

**3.** 한국어로 물었더니 전문 검색이 0건이었습니다. 왜일까요?

<details><summary>정답 보기</summary>

원문이 **영어**라 한국어 키워드가 문서에 한 글자도 없기 때문입니다. 전문 검색은 글자가 맞아야 찾습니다. 같은 질문이라도 의미 검색은 뜻으로 찾으므로 언어가 달라도 걸립니다. 단, 이것은 **다국어 임베딩 모델**을 쓸 때 이야기입니다.

</details>

---
# 5. 문서를 개체에 잇기

찾은 문서가 그래프의 개체와 이어져 있어야 검색이 문서에서 끝나지 않습니다. 그 다리를 놓습니다.

- **5-1** 이름을 `id` 로 바꾸는 사전으로 논문에서 개체를 찾습니다.
- **5-2** `MENTIONS` 로 잇고, 다리가 정말 놓였는지 검산합니다.
- **5-3** 문서에서 그래프로 한 걸음 건너가고, 이 잇기가 무엇을 놓치는지 봅니다.

## 5-1. 이름을 `id` 로 바꾸는 사전

### 왜 필요할까요?
지금까지는 **문서를 찾는** 일만 했습니다. 그런데 우리가 가진 것은 그냥 문서 더미가 아니라 **지식그래프**입니다. 논문이 말한 약물·유전자가 그래프에도 노드로 있죠.

둘을 이어 두면 이런 일이 됩니다.

> 질문으로 **논문을 찾고**, 그 논문이 언급한 **약물 노드로 건너가**, 그래프에서 그 약이 붙는 **유전자까지 따라간다.**

검색이 문서에서 끝나지 않고 **그래프로 이어지는 것**이 GraphRAG 입니다. 그 다리를 지금 놓습니다.

### 무엇을 열쇠로 잇나: 이름이 아니라 `id`
그래프의 노드는 `Compound::DB00758` 같은 **id** 로 구분합니다. 이름으로 이으면 안 됩니다. 같은 이름이 종류가 다른 노드 둘에 걸리는 경우가 이 그래프에 실제로 있습니다(`obesity` 는 질병 노드이면서 증상 노드이고, `progesterone` 은 약물이면서 약효분류입니다). 아래 사전은 애매하지 않은 이름을 **id 로 바꿔 주고**, 이렇게 **둘로 갈리는 이름 9개는 `ambiguous` 에 따로 모아 아예 잇지 않습니다.** 잘못 이으면 두 개체가 하나로 합쳐지니, 모를 때는 **안 잇는 쪽**이 안전합니다.

<img src="images/이름매칭으로_다리놓기.png" alt="논문 본문에서 사전의 이름을 찾아 id 로 바꾸고 MENTIONS 로 잇는 흐름과 그 한계" width="900">

In [ ]:
# [제공 코드] 이름 사전 불러오기: 이 셀은 실행만 하세요.
# data/name2id.json.gz 는 지식그래프의 이름과 약 이름의 동의어를 모아 둔 사전입니다.
import gzip
import json
from pathlib import Path

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
with gzip.open(DATA_DIR / "name2id.json.gz", "rt", encoding="utf-8") as _f:
    NAME2ID = json.load(_f)

for key in ["entries", "genes", "ambiguous", "brand_stopwords"]:   # 아래 함수가 쓰는 칸 넷
    print(f"  {key:16} {len(NAME2ID[key]):>6,}건")


In [ ]:
# 칸마다 한 건씩 꺼내 무엇이 들었는지 봅니다.
print("entries['aspirin']    :", NAME2ID["entries"]["aspirin"])
print("genes['CYP2C19']      :", NAME2ID["genes"]["CYP2C19"])
# 한 이름에 노드가 둘이라 어느 쪽인지 정할 수 없는 자리
print("ambiguous['estradiol']:", [item["label"] for item in NAME2ID["ambiguous"]["estradiol"]])
print("brand_stopwords       :", NAME2ID["brand_stopwords"])


| 칸 | 무엇이 들었나 |
|---|---|
| `entries` | 소문자 이름 → 그 이름이 가리키는 노드(`id`·`label`·표준 이름 `canonical`) |
| `genes` | 유전자 기호 → 노드 `id`. 기호는 대소문자가 뜻을 가르므로 소문자로 누르지 않습니다 |
| `ambiguous` | 위에서 말한 그 9개. 여기 있는 이름은 잇지 않습니다 |
| `brand_stopwords` | 흔한 영어 단어와 겹치는 상품명 8개(`today`·`correct` 등). 소문자로 쓰였으면 평범한 단어로 봅니다 |

`entries` 가 **동의어까지** 담고 있다는 데 눈길을 두세요. `aspirin` 을 넣으면 표준 이름 `Acetylsalicylic acid` 가 나옵니다. 논문마다 다르게 부르는 이름을 **한 노드로 모으려는 것**입니다.

이 네 칸을 그대로 읽어 쓰는 것이 아래 함수입니다.

In [ ]:
# [제공 코드] 이름 찾기 함수: 이 셀은 실행만 하세요.
# find_entities(문서, 사전) 가 문서에서 사전의 이름을 찾아 {노드 id: (레이블, 표준 이름)} 으로 돌려줍니다.
import re

TOKEN = re.compile(r"[A-Za-z][A-Za-z0-9'-]*")   # 한 단어의 모양: 영문으로 시작하고 숫자·따옴표·하이픈까지 한 단어로 본다


def find_entities(text, name2id):
    """문서에서 사전에 있는 이름을 찾아 {id: (레이블, 표준 이름)} 으로 돌려줍니다."""
    entries, genes = name2id["entries"], name2id["genes"]
    ambiguous, brand_stopwords = name2id["ambiguous"], name2id["brand_stopwords"]
    found = {}   # 키가 노드 id 라 같은 개체를 두 번 잡아도 한 번만 남는다
    for word in TOKEN.findall(text):
        # 유전자 기호는 대소문자를 그대로 맞춥니다(소문자로 누르면 평범한 단어가 유전자가 됩니다)
        if word in genes:
            found[genes[word]] = ("Gene", word)
            continue
        name = word.lower()   # 나머지 이름은 소문자로 맞춰 사전을 찾습니다
        # ambiguous: 종류가 다른 노드 둘에 걸리는 이름 9개. 잘못 합칠 바에는 안 잇습니다
        if name not in entries or name in ambiguous:
            continue
        # 흔한 영어 단어와 겹치는 상품명은 원래 대소문자로 쓰인 자리만 인정합니다
        if name in brand_stopwords and word.islower():
            continue
        entry = entries[name]
        found[entry["id"]] = (entry["label"], entry["canonical"])
    return found

사전이 준비됐습니다. 논문 한 편에 걸어 봅니다.

In [ ]:
# 논문 한 편에서 사전에 있는 이름을 찾아 봅니다.
sample_paper = papers[0]   # 첫 논문 한 편으로 시험해 본다
# 제목과 본문을 이어 한 덩어리로 넘긴다. 제목에만 나오는 약 이름도 잡으려고요
found = find_entities(sample_paper['title'] + ' ' + sample_paper['text'], NAME2ID)
print(sample_paper['pmcid'], '에서 찾은 개체:', len(found), '개')

In [ ]:
# 왼쪽이 레이블, 가운데가 표준 이름, 오른쪽이 그래프에서 쓰는 id
for entity_id, (label, canonical) in list(found.items())[:6]:
    print(f'  {label:20} {canonical:24} {entity_id}')

이름이 id 로 바뀌어 나옵니다.

## 5-2. `MENTIONS` 로 잇고 다리가 놓였는지 확인하기

이제 69편 전부에 사전을 걸어 `MENTIONS` 관계를 만듭니다. 만든 뒤에는 **다리가 정말 놓였는지** 검산합니다. 개체가 하나도 없는 논문이 있으면 그 논문은 검색으로 찾아도 그래프로 못 넘어갑니다.

In [ ]:
# 찾은 이름을 그래프의 개체 노드와 잇습니다. 잇는 열쇠는 이름이 아니라 id 입니다.
# 1) 69편 전부에서 개체를 찾아 레이블별로 모은다(레이블을 찍어 MATCH 해야 인덱스를 탄다)
links = {}
for paper in papers:
    for entity_id, (label, canonical) in find_entities(
            paper['title'] + ' ' + paper['text'], NAME2ID).items():
        links.setdefault(label, []).append({'pmcid': paper['pmcid'], 'id': entity_id})

# 2) 레이블별로 한 번씩 보내 MENTIONS 를 만든다. MERGE 라 같은 쌍을 두 번 이어도 관계는 하나다
for label, rows in links.items():
    run_cypher(f'UNWIND $rows AS row MATCH (d:Document {{pmcid: row.pmcid}}), '
               f'(e:{label} {{id: row.id}}) MERGE (d)-[:MENTIONS]->(e)', rows=rows)

print('MENTIONS:', run_cypher('MATCH (:Document)-[r:MENTIONS]->() '
                              'RETURN count(r) AS c')[0]['c'], '건')

In [ ]:
# 전체 건수만으론 어느 레이블에 몰렸는지 안 보인다. 레이블별로도 나눠 센다
for row in run_cypher('MATCH (:Document)-[:MENTIONS]->(e) '
                      'RETURN labels(e)[0] AS label, count(*) AS cnt ORDER BY cnt DESC'):
    print(f"  {row['label']:20} {row['cnt']:>4}")

In [ ]:
# 개체가 하나도 없는 논문은 그래프로 못 넘어간다
print('개체가 하나도 없는 논문:',
      run_cypher('MATCH (d:Document) WHERE NOT EXISTS {(d)-[:MENTIONS]->()} '
                 'RETURN count(d) AS c')[0]['c'], '편')

In [ ]:
# COUNT{...}: 그 논문에서 나가는 MENTIONS 를 세는 식. 가장 적은 논문과 가장 많은 논문을 함께 본다
span = run_cypher('MATCH (d:Document) '
                  'RETURN min(COUNT{(d)-[:MENTIONS]->()}) AS lo, '
                  'max(COUNT{(d)-[:MENTIONS]->()}) AS hi')[0]
print('한 논문이 언급한 개체 수: 최소', span['lo'], '최대', span['hi'])

69편 **전부**가 개체를 갖습니다. 어느 논문을 검색해 찾더라도 그래프로 넘어갈 다리가 있다는 뜻입니다. 다만 편차가 큽니다. 개체를 23개나 가진 논문이 있는가 하면 딱 하나만 가진 논문도 있습니다. 하나뿐인 논문은 다리가 있어도 건너가 볼 데가 좁습니다.

## 5-3. 문서에서 그래프로 한 걸음

이제 실제로 건너가 봅니다. **3-2 에서 검색으로 찾은 1위 논문**(`hits[0]`)에서 출발해, 그 논문이 언급한 약물로 가고, 다시 그 약물이 그래프에서 붙어 있는 유전자까지 두 걸음 건넙니다.

In [ ]:
# 3-2 의 hits 를 그대로 쓴다. 그 1위 논문에서 논문 -> 약물 -> 유전자로 두 걸음 건넌다
for row in run_cypher('''MATCH (d:Document {pmcid:$pmcid})-[:MENTIONS]->(c:Compound)
                         MATCH (c)-[:BINDS]->(g:Gene)
                         WITH c.name AS drug, g.name AS gene ORDER BY drug, gene
                         RETURN drug, collect(gene)[..4] AS genes
                         ORDER BY drug LIMIT 5''', pmcid=hits[0]['pmcid']):
    print(f"  {row['drug']:18} BINDS {row['genes']}")

**대부분은 논문이 적지 않은 사실입니다.** 화면에 나온 유전자는 서로 다른 것이 13개인데, 이 논문 본문에 실제로 글자가 나오는 것은 `ABCB1`·`CYP2B6`·`CYP3A4` 셋뿐입니다. 나머지 10개는 **그래프만 알던 것**이죠. 논문은 약 이름을 적었을 뿐인데, 그래프가 그 약이 붙는 유전자까지 이어 준 것입니다. 검색과 그래프가 각자 아는 것을 합친 결과입니다.

### 이 잇기는 거칩니다

지금 잇는 방법은 **사전에 있는 이름이 글자 그대로 나올 때만** 붙입니다. 그래서 이런 것을 놓칩니다.

- **두 단어 이상인 이름**은 아예 못 붙습니다. `find_entities` 가 단어를 하나씩 보기 때문에 `rheumatoid arthritis` 같은 이름은 사전에 있어도 걸리지 않습니다(사전 6,695개 중 2,555개가 그렇습니다).
- 논문이 **상품명**을 쓰거나 사전에 없는 최신 약 이름을 쓰면 못 붙습니다.
- 이름을 안 쓰고 **문장으로 돌려 말하면**("the enzyme responsible for its clearance") 못 붙습니다.
- 반대로 **다른 뜻의 같은 글자**를 개체로 잘못 잡을 수도 있습니다.

그래서 위의 `find_entities` 는 유전자 기호를 **대소문자 그대로** 맞추고, 흔한 영어 단어와 겹치는 상품명은 원래 표기로 쓰인 자리만 인정합니다. 그래도 한계는 남습니다.

> **36일차부터는 같은 일을 모델에게 시킵니다.** 문장을 읽고 개체를 뽑아내면 사전에 없는 이름과 돌려 말한 표현까지 잡을 수 있습니다. 오늘은 **다리를 놓는 데까지**입니다.

### 🖐️ 함께 따라하기: 한 유전자를 언급한 논문 찾기

다리를 **반대 방향**으로 건너 봅니다. 개체에서 출발해 그 개체를 언급한 논문을 찾는 것입니다.

1. 유전자 **`CYP3A4`** 노드를 찾아, 그 노드를 `MENTIONS` 로 가리키는 `Document` 의 `pmcid` 와 `title` 을 조회하세요(제목은 앞 50자만).
2. 몇 편인지와 함께 출력하세요.
3. 같은 기호로 4-3 의 따라하기에서 한 **전문 검색** 결과(`ft_ids`)와 편수를 비교해 보세요.

**확인 기준**: `MENTIONS` 로 **6편**이 걸리고, 4-3 의 전문 검색도 6편입니다. 다만 같은 6편이 된 이유는 두 방식이 서로 다릅니다. 전문 인덱스는 단어를 소문자로 눌러 담아 `cyp3a4` 로 찾아도 같은 6편이 나옵니다. 반대로 `find_entities` 는 유전자 기호를 **대소문자 그대로** 맞추므로, 논문이 소문자로 적었다면 `MENTIONS` 쪽만 놓쳤을 것입니다. 이 논문들이 모두 대문자로 적어서 두 답이 겹친 것입니다. 늘 같지는 않습니다. 전문 검색은 인덱스가 글자를 쪼갠 방식을 타고, `MENTIONS` 는 사전을 타기 때문입니다(4-3 의 `HLA-B` 가 그 예입니다).

In [ ]:
# 여기에 코드를 작성하세요

### ✅ 바로 확인 퀴즈 (5-1·5-3)

**1.** 문서와 개체를 이을 때 이름이 아니라 `id` 를 열쇠로 쓰는 이유는?

<details><summary>정답 보기</summary>

같은 **이름이 종류가 다른 노드 둘에 걸릴 수 있기** 때문입니다. `obesity` 는 질병 노드이면서 증상 노드이고, `progesterone` 은 약물이면서 약효분류입니다. 이름으로 이으면 이 둘이 잘못 합쳐집니다. 그래서 사전은 이런 이름을 `ambiguous` 로 빼 두고 **아예 잇지 않습니다.** 사전에서 `ambiguous` 에 든 이름을 세어 보면 이렇게 버린 이름이 몇 개인지 알 수 있습니다.

</details>

**2.** 논문이 `Coumadin`(와파린의 상품명)이라고만 적었습니다. 지금 방식으로 이어질까요?

<details><summary>정답 보기</summary>

사전에 그 상품명이 실려 있으면 이어지고, 없으면 못 이읍니다. 지금 방식은 **사전에 있는 글자만** 봅니다. 사전에 없는 이름이나 문장으로 돌려 말한 표현은 놓칩니다.

</details>

---
## 이번 강의 정리

| 절 | 문법 | 핵심 |
|---|---|---|
| 1-3 | `embed_texts` | 문장 하나가 768개 숫자. 가까움은 코사인으로 |
| 2-1 | `db.create.setNodeVectorProperty` | 벡터를 노드 속성으로 미리 적재 |
| 3-1 | `CREATE VECTOR INDEX` · `SHOW VECTOR INDEXES` | 차원 768·유사도 `cosine`, `db.awaitIndexes`. 구조는 HNSW |
| 3-2 | `SEARCH n IN (VECTOR INDEX ... FOR ... [WHERE ...] LIMIT ...)` | 뜻이 가까운 문서. 점수는 `(1+코사인)/2`. `LIMIT` 은 거르기 전에 자른다. 거를 조건은 `SEARCH` 안에(`WITH [...]` 로 등록한 속성만) |
| 4-1 | `CREATE FULLTEXT INDEX` · `db.index.fulltext.queryNodes` | 키워드 검색. 이쪽은 `SEARCH` 절이 아니라 프로시저다. 검색어는 Lucene 문법(`AND`·`OR`·`title:`) |
| 4-2 | 같은 기호를 두 검색에 나란히 | 뜻으로 올라온 문서에 그 글자가 없을 수 있다 |
| 4-3 | 검색어를 `"…"` 로 묶기 | 하이픈이 든 기호는 묶어야 쪼개진 단어가 붙어 있는 문서만 걸린다 |
| 5-1 | `find_entities` + 사전 | 이름이 아니라 **id** 로 잇는다 |
| 5-2 | `MERGE (d)-[:MENTIONS]->(e)` | 문서에서 그래프로 넘어갈 다리 |
| 5-3 | `(d)-[:MENTIONS]->(c)-[:BINDS]->(g)` | 논문이 안 적은 사실까지 그래프가 이어 준다 |

- 임베딩을 **미리 적재**하고 **벡터 인덱스**로 의미 검색합니다(차원 일치가 핵심).
- **의미 검색**은 뜻으로, **전문 검색**은 글자로. 역할이 다릅니다.
- 문서를 개체에 이어 두면 **검색이 그래프로 이어집니다.** 다만 이름 매칭은 거칩니다.

## ⏭️ 예고: 다음 시간

검색을 매번 손으로 짜는 대신 **neo4j-graphrag 의 VectorRetriever** 로 조립합니다. 거기에 33·34일차에서 배운 **PageRank·커뮤니티**를 이 그래프 위에서 다시 계산해 얹어 순위를 다듬고, 검색 결과를 **모델에 넘겨 답을 만드는** GraphRAG 까지 완성합니다. 마지막으로 자연어를 Cypher 로 바꾸는 **Text2Cypher** 개념도 살펴봅니다.

수고하셨습니다!